In [ ]:
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import Birch
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

RANDOM_STATE = 42
DATA_FILE = Path('steam-200k.csv')
if not DATA_FILE.exists():
    raise FileNotFoundError(
        'Download steam-200k.csv from https://www.kaggle.com/datasets/tamber/steam-video-games '
        'and save it beside this notebook.'
    )

In [ ]:
raw = pd.read_csv(
    DATA_FILE, header=None,
    names=['user_id', 'game_title', 'behavior', 'value', 'unused'],
    low_memory=False,
)
raw = raw.drop(columns='unused')
raw['user_id'] = raw['user_id'].astype(str).str.strip()
raw['game_title'] = raw['game_title'].astype(str).str.strip()
raw['behavior'] = raw['behavior'].astype(str).str.strip().str.lower()
raw['value'] = pd.to_numeric(raw['value'], errors='coerce')
raw = raw.dropna(subset=['value'])
raw = raw[raw['behavior'].isin(['purchase','play'])].copy()
raw = raw[(raw['user_id'] != '') & (raw['game_title'] != '')]
display(raw.head())

In [ ]:
purchases = raw.loc[raw['behavior'] == 'purchase', ['user_id', 'game_title']].drop_duplicates()
plays = (
    raw.loc[(raw['behavior'] == 'play') & (raw['value'] > 0),
            ['user_id', 'game_title', 'value']]
    .groupby(['user_id','game_title'], as_index=False)['value'].max()
)
# Count only played games for which a purchase/library flag exists.
matched_plays = plays.merge(purchases, how='inner', on=['user_id','game_title'])

owned = purchases.groupby('user_id')['game_title'].nunique().rename('games_owned')
play_stats = matched_plays.groupby('user_id').agg(
    games_played=('game_title','nunique'),
    total_hours=('value','sum'),
    max_game_hours=('value','max'),
)
profiles = owned.to_frame().join(play_stats, how='left').fillna(0)
profiles['play_ratio'] = profiles['games_played'] / profiles['games_owned']
profiles['focus_ratio'] = np.where(
    profiles['total_hours'] > 0,
    profiles['max_game_hours'] / profiles['total_hours'].replace(0, np.nan),
    0,
)
profiles['focus_ratio'] = profiles['focus_ratio'].fillna(0)
profiles['is_no_play_recorded'] = profiles['games_played'] == 0
display(profiles.describe().round(2))

In [ ]:
FEATURES = ['games_owned','games_played','total_hours','play_ratio','focus_ratio']
X_df = profiles[FEATURES].copy()
X_df[['games_owned','games_played','total_hours']] = np.log1p(
    X_df[['games_owned','games_played','total_hours']]
)
scaler = StandardScaler()
X = scaler.fit_transform(X_df)

In [ ]:
thresholds = [0.5, 0.8, 1.1]
cluster_counts = [2, 3, 4, 5, 6]
results = []

for threshold in thresholds:
    for k in cluster_counts:
        start = perf_counter()
        model = Birch(threshold=threshold, branching_factor=50, n_clusters=k)
        labels = model.fit_predict(X)
        runtime_sec = perf_counter() - start
        sizes = pd.Series(labels).value_counts()
        if len(sizes) < 2 or len(sizes) >= len(X):
            continue
        sample_n = min(1000, len(X))
        try:
            silhouette = silhouette_score(X, labels, sample_size=sample_n,
                                          random_state=RANDOM_STATE)
        except ValueError:
            silhouette = np.nan
        dbi = davies_bouldin_score(X, labels)
        results.append({
            'threshold': threshold, 'n_clusters_requested': k,
            'n_clusters_found': len(sizes),
            'branching_factor': 50,
            'silhouette': silhouette, 'davies_bouldin': dbi,
            'min_cluster_pct': (sizes.min() / len(X)) * 100,
            'subclusters': len(model.subcluster_centers_),
            'fit_seconds': runtime_sec,
        })

scores = pd.DataFrame(results)
if scores.empty:
    raise RuntimeError('No valid multi-cluster partitions found; adjust thresholds.')
scores = scores.sort_values(['silhouette','davies_bouldin'], ascending=[False,True])
display(scores.round(3))

# Avoid choosing a solution with a nearly empty cluster when possible.
acceptable = scores[(scores['min_cluster_pct'] >= 1) & scores['silhouette'].notna()]
chosen = (acceptable if not acceptable.empty else scores.dropna(subset=['silhouette'])).iloc[0]

In [ ]:
selected_k = int(chosen['n_clusters_requested'])
selected_threshold = float(chosen['threshold'])
selected_branching_factor = int(chosen['branching_factor'])

start = perf_counter()
birch = Birch(threshold=selected_threshold, branching_factor=selected_branching_factor,
              n_clusters=selected_k)
birch_labels = birch.fit_predict(X)
birch_sec = perf_counter() - start

final_sizes = pd.Series(birch_labels).value_counts()
final_model = {
    'threshold': selected_threshold,
    'branching_factor': selected_branching_factor,
    'n_clusters_requested': selected_k,
    'n_clusters_found': len(final_sizes),
    'subclusters': len(birch.subcluster_centers_),
    'silhouette': silhouette_score(X, birch_labels, sample_size=min(1000, len(X)),
                                   random_state=RANDOM_STATE),
    'davies_bouldin': davies_bouldin_score(X, birch_labels),
    'min_cluster_pct': final_sizes.min() / len(X) * 100,
    'fit_seconds': birch_sec,
}
display(pd.DataFrame([final_model]).round(3))
profiles['cluster'] = birch_labels

In [ ]:
cluster_sizes = profiles.groupby('cluster').size().rename('players')
cluster_share = (cluster_sizes / len(profiles) * 100).rename('share_pct')
cluster_medians = profiles.groupby('cluster')[FEATURES].median().round(2)
zero_play_pct = profiles.groupby('cluster')['is_no_play_recorded'].mean().mul(100).rename('no_play_record_pct')
summary = pd.concat([cluster_sizes, cluster_share, cluster_medians, zero_play_pct],axis=1).round(2)
display(summary.sort_index())

fig, ax = plt.subplots(figsize=(8, 4.5))
summary['players'].sort_index().plot(kind='bar', ax=ax)
ax.set(title='Players by BIRCH cluster', xlabel='Cluster', ylabel='Number of users')
plt.tight_layout()
plt.show()

In [ ]:
pca = PCA(n_components=2)
points = pca.fit_transform(X)
fig, ax = plt.subplots(figsize=(9, 6))
for label in sorted(set(birch_labels)):
    mask = birch_labels == label
    ax.scatter(points[mask, 0], points[mask, 1], s=8, alpha=.45, label=f'Cluster {label}')
ax.set_xlabel('Principal component 1')
ax.set_ylabel('Principal component 2')
ax.set_title('Steam player profiles grouped with BIRCH')
ax.legend(markerscale=2)
plt.tight_layout()
plt.savefig('steam_birch_pca.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
profiles.reset_index().to_csv('steam_player_profiles_with_clusters.csv', index=False)
scores.to_csv('steam_birch_parameter_results.csv', index=False)
summary.reset_index().to_csv('steam_birch_cluster_summary.csv', index=False)
pd.DataFrame([final_model]).to_csv('steam_birch_selected_model.csv', index=False)